In [2]:
from util import *
import numpy as np
import pandas as pd

In [3]:
import os

# Get common genres
print("🔍 Extracting common genres between MovieLens 1M and 100K...")
common_genres = get_common_genres(input_dir_100k='./ml-100k')

# Process MovieLens 1M
print("\n📊 Processing MovieLens 1M dataset...")
ml_1m_individual_distributions = get_individual_distributions_1m(common_genres, input_dir='./ml-1m')

# Process MovieLens 100K
print("\n📊 Processing MovieLens 100K dataset...")
ml_100k_avg_distributions = get_avg_distributions_100k(common_genres, input_dir='./ml-100k')

# Ensure consistent ordering of age groups
age_groups = sorted(ml_1m_individual_distributions.keys())

# Create ordered lists of distributions
individual_distributions_list = [
    ml_1m_individual_distributions[age_group] for age_group in age_groups
]

avg_distributions_list = [
    ml_100k_avg_distributions.get(age_group, [0] * len(common_genres))
    for age_group in age_groups
]

# 🛠 Validate size consistency
print("\n✅ Validating data consistency...")
for age_group, individuals, avg_distribution in zip(age_groups, individual_distributions_list, avg_distributions_list):
    num_individuals = len(individuals)
    num_avg_entries = len(avg_distribution)

    assert num_individuals > 0, f"❌ No individuals in age group {age_group}."
    assert num_avg_entries > 0, f"❌ No average distribution for age group {age_group}."
    assert len(individuals[0]) == num_avg_entries, (
        f"❌ Size mismatch in age group {age_group}: "
        f"Individual distributions ({len(individuals[0])}) ≠ "
        f"Average distribution ({num_avg_entries})."
    )

# Assign final variables
public_distributions = avg_distributions_list
private_distributions = individual_distributions_list
age_range_list = age_groups


print("\n🎯 Data successfully validated and ready for processing!")


🔍 Extracting common genres between MovieLens 1M and 100K...

📊 Processing MovieLens 1M dataset...
Age Group Counts:
 Age Group
25-34    395556
18-24    210747
35-44    199003
45-49     83633
50-54     72490
55+       38780
Name: count, dtype: int64

📊 Processing MovieLens 100K dataset...
Age Group Counts:
 Age Group
25-34    35444
18-24    26551
35-44    19591
45-49     6890
50-54     5890
55+       5634
Name: count, dtype: int64

✅ Validating data consistency...

🎯 Data successfully validated and ready for processing!


In [4]:
fdiv = 'tv'

In [5]:
# Initialize a dictionary to store results for the final table
from tqdm import tqdm
table_results = {}

for eps in range(5, 6):  # Example for Eps=1 and Eps=2
    results_dict = {age_range: {"Min-Max Max": 0, "Husain Max": 0} for age_range in age_range_list}
    print(f"Processing epsilon: {eps}")
    for i in tqdm(range(len(age_range_list)), desc="Processing Age Ranges"):

        age_range = age_range_list[i]
        public_distribution = public_distributions[i]
        individual_distributions = private_distributions[i]

        q = np.array(public_distribution)
        sorted_indices = np.argsort(q)
        q = q[sorted_indices]

        if len(q) == 1:
            continue

        # Compute K with prior
        K = K_with_prior(q, eps)

        min_max_max = 0
        husain_max = 0

        for p in individual_distributions:
            p = np.array(p)
            p = p[sorted_indices]

            # Perform projection
            min_max, husain = perform_projection(p, q, eps, K, fdiv)

            # Update max values
            min_max_max = max(min_max_max, min_max)
            husain_max = max(husain_max, husain)

        # Store results in the dictionary
        results_dict[age_range]["Min-Max Max"] = min_max_max
        results_dict[age_range]["Husain Max"] = husain_max

    # Populate the table results
    for age_range in age_range_list:
        if age_range not in table_results:
            table_results[age_range] = {}

        # Add results for the current epsilon
        table_results[age_range][f"Our Approach (Eps={eps})"] = round(results_dict[age_range]["Min-Max Max"],2)
        table_results[age_range][f"Husain et al. (Eps={eps})"] = round(results_dict[age_range]["Husain Max"],2)


# Convert the table results to a DataFrame
final_table_df = pd.DataFrame.from_dict(table_results, orient='index')




Processing epsilon: 5


Processing Age Ranges:   0%|          | 0/6 [00:00<?, ?it/s]

Restricted license - for non-production use only - expires 2026-11-23


Processing Age Ranges: 100%|██████████| 6/6 [00:03<00:00,  1.92it/s]
